## 5.4 Wrap-style hooks函数用法
### 5.4.1 基本用法
1. wrap_model_call

① 基于装饰器实现

我们可以同时在模型调用前后做事，所以命名为 `wrap_model_call` ，wrap意为 包裹 。

```python
def wrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]
```

（langchain 1.4.2 中 `AgentMiddleware.wrap_model_call` 的签名。返回值除了 `ModelResponse`，还可以是 `AIMessage` 或 `ExtendedModelResponse`；参数和返回值的含义见 5.4.3、5.4.4。）

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

注意: 下面的函数分别修改request和response

In [2]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    # 动态篡改用户发出的最后一条消息的内容，悄悄往里面追加字符串。
    # 典型应用：统一在底层为所有请求追加特殊的 Prompt 提示词（例如：“请用中文回答”、“禁止透漏公司机密”等）
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # 模型的调用
    # 将修改后的请求传递给 handler，真正去调用大模型（或者流转到下一个拦截器）
    # 这一步会产生真实的 Token 消耗并等待大模型响应
    response = handler(request)
    # 大模型返回响应后，在将响应交付给 Agent 状态机之前，对其内容进行直接篡改
    # `response.result` 是一个消息列表，修改其第一条返回消息的内容
    # # 典型应用：做底层的文本敏感词过滤、输出格式强行格式化、或是统一添加某些后处理标记。
    response.result[0].content += "---> wrap_model_call_after <---"

    return response

In [3]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！有什么我可以帮你？---> wrap_model_call_after <---


模型调用前消息列表的最后一条是HumanMessage，调用后最后一条是AIMessage，可以看到，模型调用前后的更改都生效了。

> ⚠️ 注意：
> 1. 返回值注解写成 `ModelResponse | None` 不准确。wrap_model_call 必须返回本次调用的结果，返回 `None` 会报 `AttributeError: 'NoneType' object has no attribute 'result'`（见 5.4.4）。
> 2. 最终 state 中的用户消息也带上了 `---> wrap_model_call_before <---`。这是因为 `request.messages[-1]` 和 state 中保存的是同一个消息对象，`+=` 直接改了它，和 07 笔记 5.3.4 直接修改 state 的问题一样。推荐用 `request.override(messages=...)` 生成新的请求，修改只对这一次模型调用生效，state 保持不变（见 5.4.4）。
> 3. `response.result[0]` 是模型刚生成、还没写入 state 的新消息，修改它能生效。但模型返回的 `content` 不一定是字符串，也可能是内容块列表（本笔记的模型带工具时就是这样），对列表 `+=` 字符串，会把字符串拆成单个字符放进列表。更稳妥的做法是用 `.text` 读取文本，再新建一条 `AIMessage`（见 5.4.4）。

② 基于类实现

In [4]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response

In [5]:
agent = create_agent(
    model=model,
    middleware=[# 为什么这里加括号?
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！有什么可以帮你？---> wrap_model_call_after <---


分析：效果和装饰器写法相同。代码注释中的问题“为什么这里加括号”：类写法定义的是类，要自己加括号创建实例；装饰器写法中，装饰器已经替我们创建好了实例。详见 07 笔记 5.3.5 第 4 点。

使用场景：用于拦截、重试、缓存模型调用。

### 5.4.2 wrap_tool_call 的用法
我们可以同时在工具调用前后做事，所以命名为 wrap_tool_call 。

① 基于装饰器实现

In [ ]:
from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call

@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result

In [7]:
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_9ae57207107d42879c524a43ed17df4c'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_9ae57207107d42879c524a43ed17df4c'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================

[{'arguments': '{"city":"北京","is_forcast":false}', 'call_id': 'call_9ae57207107d42879c524a43ed17df4c', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_a783d43eca6a06f03ca273b2582c699a', 'status': 'completed'}]
Tool Calls:
  get_weather (call_9ae57207107d42879c524a43ed17df4c)
 Call ID: call_9ae57207107d42879c524a43ed17df4c
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================

在@wrap_tool_call装饰的函数中两次调用函数并更改参数。

② 基于类实现

In [8]:

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询上海今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '上海', 'is_forcast': False}
原始参数调用结果：content='上海今天天气不错' name='get_weather' tool_call_id='call_970d331fdbe14376bc8e888fb20c0dbd'
更新以后的参数：{'city': '上海', 'is_forcast': True}
更新以后的参数调用结果：content='上海今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_970d331fdbe14376bc8e888fb20c0dbd'
================================ Human Message =================================

帮我查询上海今天的天气如何？
================================== Ai Message ==================================

[{'arguments': '{"city":"上海","is_forcast":false}', 'call_id': 'call_970d331fdbe14376bc8e888fb20c0dbd', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_380306166e083247f4a49a2ea57aa836', 'status': 'completed'}]
Tool Calls:
  get_weather (call_970d331fdbe14376bc8e888fb20c0dbd)
 Call ID: call_970d331fdbe14376bc8e888fb20c0dbd
  Args:
    city: 上海
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

上海今天天气不错
明天天气也很好
================================

使用场景：用于监控、重试、修改工具执行

> ⚠️ 注意：5.4.2 的两个示例用 `request.tool_call["args"]["is_forcast"] = True` 直接修改了工具调用的参数字典。这个字典就是 state 中 AI 消息 `tool_calls` 里的那一个，所以输出中 AI 消息的 `Args` 显示 `is_forcast: True`，而它上面一行的原始内容里，模型实际请求的是 `"is_forcast":false`：模型的请求记录被改掉了。推荐用 `request.override(tool_call=...)` 生成新的请求（见 5.4.5）。
>
> 另外，示例中 `handler` 被调用了两次，工具也真的执行了两次，最后只返回第二次的结果。查天气这类只读工具没关系；发邮件、下单这类有副作用的工具执行两次就会出问题，重复调用 `handler` 一般只用于失败后重试。

### 5.4.3 wrap hook 是怎么被调用的：包在模型调用外面

07 笔记 5.3.2 讲过：node-style hook 会变成流程图中的一个节点，框架调用它的方式是 `update = hook(state, runtime)`。wrap-style hook 不一样，它**不是节点**：`create_agent` 把 `wrap_model_call` 放进 `model` 节点内部，把 `wrap_tool_call` 放进 `tools` 节点内部，包在“真正调用模型 / 执行工具”的那一步外面。

`model` 节点的源码（`langchain/agents/factory.py`，简化）：

```python
def model_node(state, runtime):
    # ① 把 state 中的信息打包成一次“模型调用请求”
    request = ModelRequest(model=model, messages=state["messages"], system_message=system_message,
                           tools=tools, state=state, runtime=runtime, ...)

    # ② 真正调用模型的函数：取出 request 中的内容，调用模型，把结果包装成 ModelResponse
    def call_model(request):
        messages = [request.system_message, *request.messages]       # 系统提示词在这里才拼到最前面
        output = request.model.bind_tools(request.tools).invoke(messages)
        return ModelResponse(result=[output])

    # ③ 没有 wrap 中间件就直接调用；有的话，把 call_model 当作 handler 交给中间件
    if 没有 wrap_model_call 中间件:
        response = call_model(request)
    else:
        response = wrap_model_call_middleware(request, call_model)

    # ④ 把响应中的消息写回 state，这才是 model 节点的返回值
    return {"messages": response.result}
```

对照这段代码，`wrap_model_call_middleware(request, handler)` 的参数和返回值就好理解了：

| | 是什么 | 从哪来 / 到哪去 |
|---|---|---|
| `request` | 本次模型调用的“请求”：用哪个模型、发哪些消息、带哪些工具…… | 第 ① 步由 `model` 节点根据 state 创建 |
| `handler` | “继续执行”的函数，调用它才会真正调用模型 | 第 ② 步的 `call_model` |
| 返回值 | 本次调用的“响应”：模型生成的消息 | 第 ④ 步写回 state |

所以 wrap hook 是一个“拿到了调用本身”的函数：模型怎么调用、调不调用、调用几次，都由它决定。

**多个 wrap 中间件会一层套一层。** `middleware=[M1, M2]` 时，M1 拿到的 `handler` 不会直接调用模型，而是先进入 M2，M2 拿到的 `handler` 才是 `call_model`。列表中越靠前，越在外层：

```python
M1.wrap_model_call(request, handler=lambda req: M2.wrap_model_call(req, handler=call_model))
```

下面验证两点（不调用接口）：只有 wrap 中间件时，流程图中有哪些节点；wrap hook 和 before_model / after_model 的执行顺序。为了看清模型在哪一步被调用，这里使用 LangChain 自带的假模型 `FakeMessagesListChatModel`（按顺序返回预先写好的消息，不调用任何接口），并让它被调用时打印一行日志。

> 前几节用的假模型是 `GenericFakeChatModel`，这里换成了 `FakeMessagesListChatModel`。原因是：开启 LangSmith 追踪时，wrap_model_call 的追踪记录会把 `request` 中的模型对象一起序列化；`GenericFakeChatModel` 用迭代器保存预设的消息，序列化时迭代器会被读空，之后再调用模型就会报 `RuntimeError: generator raised StopIteration`。`FakeMessagesListChatModel` 用列表保存，不受影响。真实模型没有这个问题。

In [9]:
from typing import Any, Callable
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents.middleware import AgentState, ModelRequest, ModelResponse, after_model, before_model, wrap_model_call
from langgraph.runtime import Runtime


class FakeToolModel(FakeMessagesListChatModel):
    """假模型：按顺序返回预先写好的消息，不调用任何接口。"""

    def bind_tools(self, tools, **kwargs):
        return self  # create_agent 会调用 bind_tools，假模型直接返回自身


class LoggingFakeModel(FakeToolModel):
    """被调用时打印一行日志的假模型"""

    def _generate(self, *args, **kwargs):
        print("            >>> 模型被调用")
        return super()._generate(*args, **kwargs)


def make_wrap(tag: str):
    """生成一个 wrap_model_call 中间件：调用 handler 前后各打印一行"""
    @wrap_model_call(name=f"wrap_{tag}")
    def wrap(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        print(f"        {tag}：调用 handler 前")
        response = handler(request)
        print(f"        {tag}：调用 handler 后")
        return response
    return wrap


@before_model
def log_before(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("    before_model 节点")
    return None


@after_model
def log_after(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("    after_model 节点")
    return None


# ① 只有 wrap 中间件：流程图中没有新增节点
wrap_only_agent = create_agent(model=FakeToolModel(responses=[AIMessage("你好！")]),
                               middleware=[make_wrap("M1"), make_wrap("M2")])
print("只有 wrap 中间件时的节点:", list(wrap_only_agent.get_graph().nodes))

# ② 和 node-style hook 一起使用时的执行顺序
order_agent = create_agent(
    model=LoggingFakeModel(responses=[AIMessage("你好！")]),
    middleware=[log_before, make_wrap("M1"), make_wrap("M2"), log_after],
)
print("加上 node-style hook 后的节点:", list(order_agent.get_graph().nodes))
print("\n执行顺序：")
result = order_agent.invoke({"messages": [HumanMessage("你好")]})

只有 wrap 中间件时的节点: ['__start__', 'model', '__end__']
加上 node-style hook 后的节点: ['__start__', 'model', 'log_before.before_model', 'log_after.after_model', '__end__']

执行顺序：
    before_model 节点
        M1：调用 handler 前
        M2：调用 handler 前
            >>> 模型被调用
        M2：调用 handler 后
        M1：调用 handler 后
    after_model 节点


分析：

1. 只有 wrap 中间件时，流程图中仍然只有 `model` 一个节点：wrap hook 不会新增节点，它在 `model` 节点内部执行。加上 node-style hook 后，才多出 `log_before.before_model`、`log_after.after_model` 两个节点。
2. 执行顺序：before_model 节点 → 进入 `model` 节点 → M1 → M2 → 模型 → M2 → M1 → after_model 节点。M1 在外层，先进后出；“调用 handler 前”和“调用 handler 后”两段代码写在同一个函数中，中间隔着整个内层调用。

### 5.4.4 wrap_model_call 的参数和返回值

#### 1. request：本次模型调用的请求

`request` 是一个 `ModelRequest` 对象，`model` 节点每次调用模型前，都会根据 state 重新创建一个。常用属性：

| 属性 | 含义 |
|---|---|
| `model` | 本次使用的模型 |
| `messages` | 发给模型的消息列表，来自 state，不含系统提示词 |
| `system_message` | 系统提示词（`SystemMessage`），`create_agent` 的 `system_prompt` 会转成它 |
| `tools` | 本次提供给模型的工具 |
| `tool_choice` | 工具选择策略，如强制调用某个工具 |
| `response_format` | 结构化输出的格式 |
| `model_settings` | 额外传给模型的参数 |
| `state`、`runtime` | 当前的 state 和 runtime，和 node-style hook 的两个参数相同 |

下面打印一次真实调用的 `request`，并记录模型的耗时。本例调用 1 次模型。

In [10]:
import time


@wrap_model_call
def inspect_request(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    # 调用前：查看请求
    print("model          :", type(request.model).__name__, request.model.model_name)
    print("messages       :", [(type(m).__name__, m.text) for m in request.messages])
    print("system_message :", request.system_message.text)
    print("tools          :", [t.name for t in request.tools])
    print("state 的键     :", list(request.state.keys()))
    print("runtime        :", type(request.runtime).__name__)

    start = time.time()
    response = handler(request)                       # 真正调用模型
    elapsed = time.time() - start

    # 调用后：查看响应。start 是局部变量，调用前后的代码可以直接共用
    print(f"\n模型耗时 {elapsed:.1f} 秒")
    print("handler 返回   :", type(response).__name__)
    print("response.result:", [(type(m).__name__, m.text) for m in response.result])
    return response


inspect_agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="你是一个天气助手，回答尽量简短。",
    middleware=[inspect_request],
)
result = inspect_agent.invoke({"messages": [HumanMessage("你好")]})

model          : ChatOpenAI gpt-6-luna
messages       : [('HumanMessage', '你好')]
system_message : 你是一个天气助手，回答尽量简短。
tools          : ['get_weather']
state 的键     : ['messages']
runtime        : Runtime

模型耗时 4.9 秒
handler 返回   : ModelResponse
response.result: [('AIMessage', '你好，需要我帮你查询天气，还是处理代码问题？')]


分析：

1. `messages` 中只有用户消息，系统提示词单独放在 `system_message` 中，真正调用模型时才拼到最前面（第七章 04 笔记 7.1.1）。
2. `handler(request)` 返回的是 `ModelResponse`，模型生成的消息在它的 `result` 列表中。
3. 调用前记下的 `start`，调用后可以直接使用。node-style hook 要做同样的事，得在 before_model 中把开始时间存进 state，再在 after_model 中读出来（5.4.6）。

#### 2. 修改 request：用 override，只影响这一次调用

`request` 是 `model` 节点每次临时创建的。想修改它，官方的写法是 `request.override(属性=新值)`：它返回一个修改后的**新请求**，原来的 `request` 不变。把新请求交给 `handler`，修改就只作用于这一次模型调用，**不会写进 state**。

三种修改方式的对比（均在 langchain 1.4.2 中实测）：

| 写法 | 结果 |
|---|---|
| `handler(request.override(messages=新列表))` | ✅ 推荐。模型收到修改后的内容，state 不变 |
| `request.messages = 新列表` | 能生效，但会报 `DeprecationWarning`：直接给 request 的属性赋值已不推荐，应改用 `override` |
| `request.messages[-1].content += "..."` | 能生效，但改的是 state 中那条消息对象本身，修改会留在 state 中，连调用方传入的消息对象也会被改（5.4.1 的输出就是这样） |

常见用法（官方文档的例子）：临时修改系统提示词 `override(system_message=...)`、切换模型 `override(model=...)`、筛选工具 `override(tools=...)`。

下面把 5.4.1 的例子改成 `override` 写法：在问题后面临时加上“请用一句话回答”。本例调用 1 次模型。

In [11]:
@wrap_model_call
def answer_in_one_sentence(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    last = request.messages[-1]
    new_last = HumanMessage(content=last.text + "（请用一句话回答）")        # 新建消息，不改原来的对象
    new_request = request.override(messages=[*request.messages[:-1], new_last])
    print("模型收到的最后一条消息:", new_request.messages[-1].text)
    return handler(new_request)


override_agent = create_agent(model=model, middleware=[answer_in_one_sentence])
result = override_agent.invoke({"messages": [HumanMessage("介绍一下 LangChain")]})

print("\nstate 中的消息:")
for msg in result["messages"]:
    print(f"    {type(msg).__name__}: {msg.text}")

模型收到的最后一条消息: 介绍一下 LangChain（请用一句话回答）

state 中的消息:
    HumanMessage: 介绍一下 LangChain
    AIMessage: LangChain 是一个用于构建大语言模型应用的开发框架，帮助开发者连接模型、数据源和工具，并编排复杂任务流程。


分析：模型收到的问题带上了“请用一句话回答”，回答也确实只有一句；但 state 中的用户消息还是原来的“介绍一下 LangChain”。对比 5.4.1：那里直接修改了消息对象，state 中的用户消息也被改了。

如果确实要把修改保存到 state 中，应该用 before_model（07 笔记 5.3.4，通过返回值修改 state），或者用本节第 4 点的 `ExtendedModelResponse`。

#### 3. handler：可以调用 0 次、1 次或多次

`handler` 就是 5.4.3 中“继续执行”的那个函数：调用 `handler(request)`，请求才会继续传给内层的中间件，最终到达模型；模型的返回值、模型抛出的异常，也都从 `handler(request)` 这里回到我们手中。所以 wrap hook 可以决定：

| 调用次数 | 效果 | 典型用途 |
|---|---|---|
| 0 次 | 不调用模型，直接返回一个结果（短路） | 缓存、拦截 |
| 1 次 | 正常调用 | 修改请求或响应、记录日志和耗时 |
| 多次 | 用 `try/except` 包住 `handler(request)`，失败了再调用 | 重试、失败后换模型（内置的 `ModelRetryMiddleware`、`ModelFallbackMiddleware` 就是这样实现的） |

**示例 1：缓存（0 次）。** 同一个问题第二次出现时，直接返回上次的回答，不调用模型。本例调用 1 次模型：第 2 次 `invoke` 命中缓存。

In [12]:
answer_cache = {}   # 问题 → 回答


@wrap_model_call
def cache_answer(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | AIMessage:
    question = request.messages[-1].text
    if question in answer_cache:
        print(f"命中缓存，不调用模型：{question}")
        return AIMessage(content=answer_cache[question])        # 直接返回 AIMessage，框架会自动包装成 ModelResponse

    print(f"没有缓存，调用模型：{question}")
    response = handler(request)
    answer_cache[question] = response.result[0].text
    return response


cache_agent = create_agent(model=model, middleware=[cache_answer])
for i in range(2):
    result = cache_agent.invoke({"messages": [HumanMessage("用一句话介绍 Python")]})
    print(f"第 {i + 1} 次回答：{result['messages'][-1].text}\n")

没有缓存，调用模型：用一句话介绍 Python
第 1 次回答：Python 是一种简洁易学、用途广泛的编程语言，常用于自动化、数据分析、人工智能和 Web 开发。

命中缓存，不调用模型：用一句话介绍 Python
第 2 次回答：Python 是一种简洁易学、用途广泛的编程语言，常用于自动化、数据分析、人工智能和 Web 开发。



**示例 2：重试（多次）。** 真实模型很难按需出错，这里用一个内层中间件 `flaky_network` 模拟网络故障：第 1 次调用直接抛出异常，之后正常调用模型。外层的 `retry_model` 捕获异常后，再调用一次 `handler`。本例调用 1 次模型：第 1 次在到达模型之前就失败了。

In [13]:
attempt_count = {"n": 0}


@wrap_model_call
def flaky_network(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """模拟网络故障：第 1 次调用直接抛出异常"""
    attempt_count["n"] += 1
    if attempt_count["n"] == 1:
        raise ConnectionError("模拟网络错误")
    return handler(request)


@wrap_model_call
def retry_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """失败后重试，最多调用 3 次"""
    for attempt in range(1, 4):
        try:
            return handler(request)
        except Exception as e:
            print(f"第 {attempt} 次调用失败：{e!r}")
            if attempt == 3:
                raise


retry_agent = create_agent(model=model, middleware=[retry_model, flaky_network])   # retry_model 在外层
result = retry_agent.invoke({"messages": [HumanMessage("用一句话介绍 Python")]})
print("最终回答：", result["messages"][-1].text)
print("retry_model 一共调用了", attempt_count["n"], "次 handler")

第 1 次调用失败：ConnectionError('模拟网络错误')
最终回答： Python 是一种简洁易学、功能强大且广泛应用于数据分析、人工智能、Web 开发和自动化等领域的高级编程语言。
retry_model 一共调用了 2 次 handler


分析：

1. 缓存：第 2 次 `invoke` 没有调用 `handler`，直接返回了 `AIMessage`，模型没有被调用，回答和第 1 次完全相同。
2. 重试：`retry_model` 在外层，它的 `handler` 包含了内层的 `flaky_network` 和模型（5.4.3 的嵌套）。第 1 次调用时 `flaky_network` 抛出的异常，从 `retry_model` 的 `handler(request)` 处抛了出来，被 `try/except` 捕获后再调用一次，第 2 次成功。
3. 能捕获异常，是因为模型调用发生在 `handler(request)` 这一行，就在 wrap hook 的函数体内。node-style hook 做不到：模型在另一个节点中调用，出错时整个 Agent 直接报错（5.4.6）。

#### 4. 返回值：ModelResponse、AIMessage 或 ExtendedModelResponse

返回值就是本次模型调用的结果，由 `model` 节点写回 state（5.4.3 第 ④ 步）。可以返回三种类型：

| 返回值 | 说明 |
|---|---|
| `ModelResponse` | 最常见：直接返回 `handler(request)` 的结果，或者新建一个。它有两个属性：`result`（消息列表，通常只有一条 `AIMessage`）和 `structured_response`（结构化输出的结果，没有时为 `None`） |
| `AIMessage` | 简写：只返回一条消息，框架自动包装成 `ModelResponse(result=[消息])`。上面的缓存示例就是这样 |
| `ExtendedModelResponse` | 了解：在响应之外再附带一个 `Command`，用来更新 state 中的其他字段，见下方代码 |

不能返回 `None`：忘了写 `return`，或者像 5.4.1 的注解那样以为可以返回 `None`，都会报 `AttributeError: 'NoneType' object has no attribute 'result'`。

**修改模型的回复**时，推荐新建消息，而不是直接修改 `response.result[0].content`（原因见 5.4.1 后面的注意第 3 点）。下面的例子改编自官方文档：

```python
@wrap_model_call
def uppercase_responses(request, handler):
    response = handler(request)
    ai_msg = response.result[0]
    return ModelResponse(
        result=[AIMessage(content=ai_msg.text.upper())],          # 新建一条 AIMessage
        structured_response=response.structured_response,        # 结构化输出原样保留
    )
```

**ExtendedModelResponse**（了解）：wrap hook 修改 request 不会写进 state；如果确实要在这里更新 state，就在返回值中附带一个 `Command(update={...})`。字段要先用 `state_schema` 声明（08 笔记 5.3.9）：

```python
from typing_extensions import NotRequired
from langgraph.types import Command
from langchain.agents.middleware import AgentState, ExtendedModelResponse


class UsageState(AgentState):
    model_calls: NotRequired[int]


@wrap_model_call(state_schema=UsageState)
def count_model_calls(request, handler):
    response = handler(request)
    return ExtendedModelResponse(
        model_response=response,
        command=Command(update={"model_calls": request.state.get("model_calls", 0) + 1}),
    )
```

### 5.4.5 wrap_tool_call 的参数和返回值

`wrap_tool_call` 的道理和 `wrap_model_call` 一样，只是包的是工具执行：`tools` 节点为模型请求的**每一个**工具调用创建一个 `ToolCallRequest`，把“真正执行工具”的函数作为 `handler` 交给中间件，再把返回的 `ToolMessage` 写回 state。模型一次请求了两个工具调用，wrap_tool_call 就会执行两次。

```python
def wrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
) -> ToolMessage | Command[Any]
```

| | 说明 |
|---|---|
| `request.tool_call` | 模型发起的这一个工具调用，是字典 `{"name": 工具名, "args": 参数字典, "id": 调用 id}` |
| `request.tool` | 要执行的工具对象（`BaseTool`） |
| `request.state`、`request.runtime` | 当前的 state 和工具的运行时信息（`ToolRuntime`，了解即可） |
| `handler(request)` | 真正执行工具，返回 `ToolMessage`；工具抛出的异常也会从这里抛出 |
| 返回值 | `ToolMessage`（工具结果）或 `Command`（了解）。不能返回 `None`，否则报 `NotImplementedError: Unsupported message type: <class 'NoneType'>` |

> `ToolCallRequest` 可以从 `langchain.agents.middleware` 导入，和 5.4.2 中从 `langgraph.prebuilt.tool_node` 导入的是同一个类。

修改工具调用同样要用 `override`，先复制一份字典再改：

```python
new_call = {**request.tool_call, "args": {**request.tool_call["args"], "is_forcast": True}}
return handler(request.override(tool_call=new_call))
```

下面验证（不调用接口）：用 `override` 修改参数后，state 中模型的请求记录保持不变。为了让模型一定一次发起两个工具调用，这里使用 5.4.3 的假模型。

In [14]:
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain.agents.middleware import ToolCallRequest, wrap_tool_call


@wrap_tool_call
def force_forecast(request: ToolCallRequest,
                   handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]) -> ToolMessage | Command[Any]:
    print(f"wrap_tool_call 处理 {request.tool_call['id']}：{request.tool.name}({request.tool_call['args']})")
    new_call = {**request.tool_call, "args": {**request.tool_call["args"], "is_forcast": True}}   # 复制一份再改
    return handler(request.override(tool_call=new_call))


tool_model = FakeToolModel(responses=[
    AIMessage("", tool_calls=[
        {"name": "get_weather", "args": {"city": "北京", "is_forcast": False}, "id": "call_1"},
        {"name": "get_weather", "args": {"city": "上海", "is_forcast": False}, "id": "call_2"},
    ]),
    AIMessage("北京和上海今天天气都不错，明天也很好。"),
])
tool_agent = create_agent(model=tool_model, tools=[get_weather], middleware=[force_forecast])
result = tool_agent.invoke({"messages": [HumanMessage("北京和上海天气怎么样？")]})

print("\nstate 中模型的请求：", [call["args"] for call in result["messages"][1].tool_calls])
print("工具结果：", [msg.text for msg in result["messages"] if isinstance(msg, ToolMessage)])

wrap_tool_call 处理 call_1：get_weather({'city': '北京', 'is_forcast': False})
wrap_tool_call 处理 call_2：get_weather({'city': '上海', 'is_forcast': False})

state 中模型的请求： [{'city': '北京', 'is_forcast': False}, {'city': '上海', 'is_forcast': False}]
工具结果： ['北京今天天气不错\n明天天气也很好', '上海今天天气不错\n明天天气也很好']


分析：

1. 模型一次请求了两个工具调用，`force_forecast` 执行了两次，每次处理一个。
2. 两个工具都按 `is_forcast=True` 执行了（结果中有明天的天气），但 state 中模型的请求仍然是 `is_forcast: False`，请求记录没有被改。对比 5.4.2：直接修改字典后，AI 消息中的参数也变成了 `True`。

**短路与异常处理。** 和 `wrap_model_call` 一样，可以不调用 `handler` 直接返回结果，也可以用 `try/except` 处理工具抛出的异常。`create_agent` 默认只把“参数不合法”这类错误转成消息交给模型，工具执行时抛出的异常会直接中断整个 Agent（01 笔记 1.4.5）；用 wrap_tool_call 把异常转成 `ToolMessage`，模型就能看到错误并继续回答。

下面用假模型一次请求两个工具：一个没有权限调用，一个执行时会出错（不调用接口）。

In [15]:
@tool
def delete_file(path: str) -> str:
    """删除指定文件"""
    return f"已删除 {path}"


@tool
def get_stock_price(code: str) -> str:
    """查询股票价格"""
    raise TimeoutError("行情接口超时")


BLOCKED_TOOLS = {"delete_file"}


@wrap_tool_call
def guard_tools(request: ToolCallRequest,
                handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]) -> ToolMessage | Command[Any]:
    name = request.tool_call["name"]
    if name in BLOCKED_TOOLS:                       # 0 次：不执行工具，直接返回结果
        return ToolMessage(content=f"没有权限调用 {name}", tool_call_id=request.tool_call["id"], status="error")
    try:                                            # 1 次：执行工具，出错时把异常转成工具结果
        return handler(request)
    except Exception as e:
        return ToolMessage(content=f"{name} 执行失败：{e}", tool_call_id=request.tool_call["id"], status="error")


guard_model = FakeToolModel(responses=[
    AIMessage("", tool_calls=[
        {"name": "delete_file", "args": {"path": "/data/report.txt"}, "id": "call_1"},
        {"name": "get_stock_price", "args": {"code": "600519"}, "id": "call_2"},
    ]),
    AIMessage("没有权限删除文件；股价暂时查不到，请稍后再试。"),
])
guard_agent = create_agent(model=guard_model, tools=[delete_file, get_stock_price], middleware=[guard_tools])
result = guard_agent.invoke({"messages": [HumanMessage("删除 report.txt，再查一下 600519 的股价")]})

for msg in result["messages"]:
    if isinstance(msg, ToolMessage):
        print(f"ToolMessage(status={msg.status}): {msg.text}")
print("最终回答:", result["messages"][-1].text)

ToolMessage(status=error): 没有权限调用 delete_file
ToolMessage(status=error): get_stock_price 执行失败：行情接口超时
最终回答: 没有权限删除文件；股价暂时查不到，请稍后再试。


分析：`delete_file` 没有执行（短路），`get_stock_price` 的异常被转成了 `status="error"` 的工具结果，Agent 正常结束。不加这个中间件时，`get_stock_price` 的异常会直接中断整个 Agent。（最终回答是假模型预先写好的；换成真实模型，它会根据这两条错误信息组织回答。）

### 5.4.6 为什么 wrap 风格和节点风格的定义方法不同

根本原因是两者**拿到的东西不同**：node-style hook 是流程图中的一个节点，拿到的是两个步骤之间的 state；wrap-style hook 在节点内部，拿到的是“调用”本身（`request` + `handler`）。要控制一次调用（不调用、调用多次、捕获它的异常），就必须拿到这次调用，所以参数只能是“请求 + 继续执行的函数”，返回值只能是“这次调用的结果”。

| | node-style（如 `before_model`） | wrap-style（如 `wrap_model_call`） |
|---|---|---|
| 在流程图中的位置 | 单独的节点（07 笔记 5.3.2） | 不是节点，在 `model` / `tools` 节点内部执行（5.4.3） |
| 参数 | `state`、`runtime` | `request`、`handler` |
| 返回值 | 对 state 的修改：字典或 `None` | 本次调用的结果：`ModelResponse` / `AIMessage` / `ToolMessage`，不能是 `None` |
| 修改的对象 | state，修改会保存下来 | request，只影响这一次调用（要保存需用 `ExtendedModelResponse`） |
| 能否决定模型 / 工具调不调用 | 不能（只能用 `jump_to` 跳过，08 笔记 5.3.8） | 能：调用 0 次、1 次或多次 |
| 能否捕获模型 / 工具的异常 | 不能，模型在另一个节点中调用 | 能，异常从 `handler(request)` 处抛出 |
| 调用前后的逻辑 | 分成两个 hook（before_model、after_model），要共享数据只能存进 state | 写在同一个函数中，用局部变量共享（如 5.4.4 的计时） |
| 多个中间件的顺序 | before 正序、after 倒序（07 笔记 5.3.6） | 一层套一层，列表中第一个在最外层 |
| 只写同步版本、用 `ainvoke` 调用 | 自动使用同步版本（07 笔记 5.3.5） | 报 `NotImplementedError`，需要另外实现 `awrap_model_call` |
| 适合 | 记录日志、校验、更新 state | 重试、换模型、缓存、临时修改提示词 / 模型 / 工具、异常处理 |

> 官方文档的建议也是这样分工：顺序执行的逻辑（日志、校验）用 node-style，控制流程的逻辑（重试、回退、缓存）用 wrap-style。

两点补充：

1. **为什么 wrap 风格没有 `can_jump_to`**：它不是节点，没有出边可选；想跳过模型，不调用 `handler` 就行（08 笔记 5.3.9 第 5 点）。
2. **两种写法都能实现时怎么选**：需要修改并保存对话内容（如压缩历史消息的 `SummarizationMiddleware`），用 node-style 返回字典；只想影响这一次调用（如临时加提示词、换模型），用 wrap-style 的 `override`。

### 5.4.7 小结

**wrap_model_call 的写法模板：**

```python
@wrap_model_call
def my_wrap(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    # ① 调用前：读取 request；要修改就用 request.override(...) 生成新请求
    response = handler(request)          # ② 调用：0 次（短路）、1 次或多次（重试）
    # ③ 调用后：读取或替换 response
    return response                      # ④ 必须返回结果，不能返回 None
```

`wrap_tool_call` 同理，只是 `request` 换成 `ToolCallRequest`，返回 `ToolMessage`。

**常见错误（langchain 1.4.2 实测）：**

| 错误写法 | 结果 |
|---|---|
| wrap_model_call 返回 `None`（如忘了写 `return`） | `AttributeError: 'NoneType' object has no attribute 'result'` |
| wrap_tool_call 返回 `None` | `NotImplementedError: Unsupported message type: <class 'NoneType'>` |
| 直接修改 `request.messages[-1].content` | 能生效，但 state 中的消息、调用方传入的消息对象也被改了 |
| 直接修改 `request.tool_call["args"]` | 能生效，但 state 中 AI 消息记录的工具参数也被改了 |
| 给 request 的属性赋值：`request.messages = [...]` | 能生效，但报 `DeprecationWarning`，应改用 `request.override(...)` |
| 对 `response.result[0].content` 做 `+=` | content 是内容块列表时，字符串会被拆成单个字符放进列表 |
| 只写了同步的 `wrap_model_call`，用 `ainvoke` / `astream` 调用 | `NotImplementedError: Asynchronous implementation of awrap_model_call is not available...` |